# 03 — Training
**Project**: YOLOv11s Person Detection  
**Tujuan**: Training YOLOv11s dengan transfer learning pada dataset person COCO 2017

---

## 3.1 Import Libraries & Verifikasi GPU

In [31]:
import torch
import yaml
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib_inline
from pathlib import Path
from ultralytics import YOLO

# ── Verifikasi GPU ─────────────────────────────────────────────────────────────
print('=== Environment Check ===')
print(f'PyTorch version  : {torch.__version__}')
print(f'CUDA available   : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'CUDA version     : {torch.version.cuda}')
    print(f'GPU name         : {torch.cuda.get_device_name(0)}')
    print(f'VRAM total       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
    DEVICE = 0
else:
    print('WARNING: CUDA tidak tersedia! Training akan berjalan di CPU (sangat lambat).')
    DEVICE = 'cpu'

print(f'\nDevice yang digunakan: {DEVICE}')

=== Environment Check ===
PyTorch version  : 2.9.1+cu126
CUDA available   : True
CUDA version     : 12.6
GPU name         : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM total       : 8.59 GB

Device yang digunakan: 0


## 3.2 Konfigurasi Training Variables

> Semua hyperparameter didefinisikan di sini — ubah di satu tempat ini saja.

In [25]:
# ── Path ──────────────────────────────────────────────────────────────────────
DATASET_YAML   = 'dataset/dataset.yaml'
MODEL_WEIGHTS  = 'yolo11s.pt'          # Pretrained YOLOv11 Small
PROJECT_DIR    = 'runs/train'
RUN_NAME       = 'person_yolo11s_v1'

# ── Training hyperparameter ───────────────────────────────────────────────────
EPOCHS         = 50       # Jumlah epoch maksimum
IMGSZ          = 640      # Ukuran input gambar (px) — standar YOLO
BATCH          = 16       # Batch size — turunkan ke 8 jika VRAM < 8 GB
WORKERS        = 4        # DataLoader workers

# ── Optimizer ─────────────────────────────────────────────────────────────────
OPTIMIZER      = 'SGD'    # Optimizer: SGD (default YOLO, stabil) atau AdamW
LR0            = 0.01     # Learning rate awal
LRF            = 0.01     # Learning rate final factor (lr_final = lr0 * lrf)
MOMENTUM       = 0.937    # SGD momentum
WEIGHT_DECAY   = 0.0005   # Regularisasi L2
WARMUP_EPOCHS  = 3.0      # Epoch warmup (lr naik perlahan dari 0)
WARMUP_MOMENTUM= 0.8      # Momentum awal saat warmup

# ── Early Stopping & Saving ───────────────────────────────────────────────────
PATIENCE       = 10       # Stop jika val/mAP50 tidak membaik dalam N epoch
SAVE_PERIOD    = 10       # Simpan checkpoint setiap N epoch

# ── Loss weights ──────────────────────────────────────────────────────────────
BOX            = 7.5      # Box regression loss weight
CLS            = 0.5      # Classification loss weight
DFL            = 1.5      # Distribution Focal Loss weight

# ── Augmentasi (built-in Ultralytics) ─────────────────────────────────────────
MOSAIC         = 1.0      # Probabilitas mosaic augmentation (4 gambar digabung)
MIXUP          = 0.0      # MixUp augmentation (0 = nonaktif)
COPY_PASTE     = 0.0      # Copy-Paste augmentation
DEGREES        = 5.0      # Rotasi acak ± N derajat
TRANSLATE      = 0.1      # Translasi acak (fraksi ukuran gambar)
SCALE          = 0.5      # Scale acak (zoom in/out)
SHEAR          = 0.0      # Shear acak (derajat)
PERSPECTIVE    = 0.0      # Perspective transform
FLIPUD         = 0.0      # Flip vertikal (tidak relevan untuk person detection)
FLIPLR         = 0.5      # Flip horizontal
HSV_H          = 0.015    # Hue shift
HSV_S          = 0.7      # Saturation shift
HSV_V          = 0.4      # Value/Brightness shift
ERASING        = 0.4      # Random erasing (regularisasi)

# ── Inference threshold (saat val) ────────────────────────────────────────────
CONF_THRESH    = 0.25     # Confidence threshold
IOU_THRESH     = 0.5      # IoU threshold untuk NMS

# ── Random seed ───────────────────────────────────────────────────────────────
SEED           = 42

print('Training Variables:')
train_vars = {
    'model'        : MODEL_WEIGHTS,
    'epochs'       : EPOCHS,
    'imgsz'        : IMGSZ,
    'batch'        : BATCH,
    'optimizer'    : OPTIMIZER,
    'lr0'          : LR0,
    'lrf'          : LRF,
    'momentum'     : MOMENTUM,
    'weight_decay' : WEIGHT_DECAY,
    'patience'     : PATIENCE,
    'device'       : DEVICE,
    'mosaic'       : MOSAIC,
    'fliplr'       : FLIPLR,
    'conf'         : CONF_THRESH,
}
for k, v in train_vars.items():
    print(f'  {k:15s}: {v}')

Training Variables:
  model          : yolo11s.pt
  epochs         : 50
  imgsz          : 640
  batch          : 16
  optimizer      : SGD
  lr0            : 0.01
  lrf            : 0.01
  momentum       : 0.937
  weight_decay   : 0.0005
  patience       : 10
  device         : 0
  mosaic         : 1.0
  fliplr         : 0.5
  conf           : 0.25


## 3.3 Verifikasi Dataset YAML

In [14]:
with open(DATASET_YAML) as f:
    dataset_cfg = yaml.safe_load(f)

print('Isi dataset.yaml:')
for k, v in dataset_cfg.items():
    print(f'  {k}: {v}')

# Verifikasi direktori
base = Path(dataset_cfg['path'])
for split in ['train', 'val', 'test']:
    key = split
    split_path = base / dataset_cfg[key]
    n_files = len(list(split_path.glob('*')))
    print(f'  {split}: {split_path} → {n_files} file')

print(f'\nJumlah class: {dataset_cfg["nc"]}')
print(f'Nama class  : {dataset_cfg["names"]}')

Isi dataset.yaml:
  path: C:\NaufalFirdaus\CODES\AI\person-tracker-yolov11\files\dataset
  train: images/train
  val: images/val
  test: images/test
  nc: 1
  names: ['person']
  train: C:\NaufalFirdaus\CODES\AI\person-tracker-yolov11\files\dataset\images\train → 2100 file
  val: C:\NaufalFirdaus\CODES\AI\person-tracker-yolov11\files\dataset\images\val → 450 file
  test: C:\NaufalFirdaus\CODES\AI\person-tracker-yolov11\files\dataset\images\test → 450 file

Jumlah class: 1
Nama class  : ['person']


## 3.4 Load Pretrained Model

In [15]:
model = YOLO(MODEL_WEIGHTS)

print(f'Model berhasil di-load: {MODEL_WEIGHTS}')
print(f'Model type   : {type(model)}')
print(f'\nModel info:')
model.info()

Model berhasil di-load: yolo11s.pt
Model type   : <class 'ultralytics.models.yolo.model.YOLO'>

Model info:
YOLO11s summary: 181 layers, 9,458,752 parameters, 0 gradients, 21.7 GFLOPs


(181, 9458752, 0, 21.718374400000002)

## 3.5 Baseline Evaluation (Pre-training)

Evaluasi performa model **sebelum** fine-tuning. Ini menjadi baseline perbandingan.

In [16]:
print('Menjalankan baseline evaluation (sebelum fine-tuning)...')

baseline_metrics = model.val(
    data=DATASET_YAML,
    split='val',
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    conf=CONF_THRESH,
    iou=IOU_THRESH,
    verbose=False
)

baseline_map50    = baseline_metrics.box.map50
baseline_map5095  = baseline_metrics.box.map
baseline_precision = baseline_metrics.box.mp
baseline_recall    = baseline_metrics.box.mr

print('\n=== Baseline (Pretrained, Belum Fine-tuned) ===')
print(f'  mAP@0.5      : {baseline_map50:.4f}')
print(f'  mAP@0.5:0.95 : {baseline_map5095:.4f}')
print(f'  Precision    : {baseline_precision:.4f}')
print(f'  Recall       : {baseline_recall:.4f}')

# Simpan baseline
baseline_results = {
    'map50'    : float(baseline_map50),
    'map5095'  : float(baseline_map5095),
    'precision': float(baseline_precision),
    'recall'   : float(baseline_recall)
}
import json
Path('runs').mkdir(exist_ok=True)
with open('runs/baseline_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)
print('\nBaseline tersimpan ke runs/baseline_results.json')

Menjalankan baseline evaluation (sebelum fine-tuning)...
Ultralytics 8.4.6  Python-3.12.10 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11s summary (fused): 100 layers, 9,443,760 parameters, 0 gradients, 21.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1461.9416.9 MB/s, size: 172.3 KB)
val: Scanning C:\NaufalFirdaus\CODES\AI\person-tracker-yolov11\files\dataset\labels\val.cache... 450 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 450/450  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 5.5it/s 5.2s0.1s
                   all        450       1909      0.839      0.729      0.821      0.643
Speed: 1.0ms preprocess, 7.5ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to C:\NaufalFirdaus\CODES\AI\person-tracker-yolov11\files\runs\detect\val2

=== Baseline (Pretrained, Belum Fine-tuned) ===
  mAP@0.5      : 0.8213
  mAP@0.5:0.95 : 0.6427
  Precision    : 

## 3.6 Training

> Training dimulai. Monitor output di bawah.
> Estimasi waktu bergantung pada GPU — dengan RTX 3060 biasanya ~30–60 menit untuk 50 epoch.

In [17]:
import time

print('Memulai training...')
start_time = time.time()

results = model.train(
    # Data
    data          = DATASET_YAML,
    # Model & ukuran
    imgsz         = IMGSZ,
    # Training loop
    epochs        = EPOCHS,
    batch         = BATCH,
    workers       = WORKERS,
    # Optimizer
    optimizer     = OPTIMIZER,
    lr0           = LR0,
    lrf           = LRF,
    momentum      = MOMENTUM,
    weight_decay  = WEIGHT_DECAY,
    warmup_epochs = WARMUP_EPOCHS,
    warmup_momentum = WARMUP_MOMENTUM,
    # Loss weights
    box           = BOX,
    cls           = CLS,
    dfl           = DFL,
    # Augmentasi
    mosaic        = MOSAIC,
    mixup         = MIXUP,
    copy_paste    = COPY_PASTE,
    degrees       = DEGREES,
    translate     = TRANSLATE,
    scale         = SCALE,
    shear         = SHEAR,
    perspective   = PERSPECTIVE,
    flipud        = FLIPUD,
    fliplr        = FLIPLR,
    hsv_h         = HSV_H,
    hsv_s         = HSV_S,
    hsv_v         = HSV_V,
    erasing       = ERASING,
    # Stopping & saving
    patience      = PATIENCE,
    save_period   = SAVE_PERIOD,
    # Hardware
    device        = DEVICE,
    # Reproducibility
    seed          = SEED,
    # Output
    project       = PROJECT_DIR,
    name          = RUN_NAME,
    exist_ok      = False,
    # Verbose
    verbose       = True,
)

elapsed = time.time() - start_time
print(f'\nTraining selesai dalam {elapsed/60:.1f} menit')

Memulai training...
New https://pypi.org/project/ultralytics/8.4.36 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.6  Python-3.12.10 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/dataset.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, nam

## 3.7 Hasil Training — Loss & Metrics Curves

In [19]:
run_dir = Path(PROJECT_DIR) / RUN_NAME

# Auto-detect: jika path tidak ditemukan, cari results.csv di seluruh runs/
if not run_dir.exists():
    found = list(Path('runs').rglob(f'{RUN_NAME}/results.csv'))
    if found:
        run_dir = found[0].parent
        print(f'[INFO] run_dir dikoreksi -> {run_dir}')
    else:
        raise FileNotFoundError(f'Tidak ditemukan run {RUN_NAME}. Pastikan Cell training sudah dijalankan.')

results_csv = run_dir / 'results.csv'

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

print(f'Training selesai pada epoch: {len(df)}')
print(f'Run dir: {run_dir}')
print(f'\nKolom tersedia: {list(df.columns)}')
df.tail(5)

[INFO] run_dir dikoreksi -> runs\detect\runs\train\person_yolo11s_v1
Training selesai pada epoch: 50
Run dir: runs\detect\runs\train\person_yolo11s_v1

Kolom tersedia: ['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
45,46,1581.62,0.89900,0.57915,1.00672,0.78002,0.65569,0.73058,0.47705,1.20562,0.81921,1.15185,0.001090,0.001090,0.001090
46,47,1613.25,0.89041,0.56693,0.99900,0.76202,0.65585,0.72417,0.47721,1.19600,0.82625,1.14903,0.000892,0.000892,0.000892
47,48,1644.83,0.89112,0.57180,0.99879,0.74885,0.67103,0.72982,0.47810,1.20064,0.82751,1.14628,0.000694,0.000694,0.000694
48,49,1676.47,0.87949,0.55756,1.00081,0.75452,0.66737,0.72864,0.47802,1.19333,0.81573,1.14275,0.000496,0.000496,0.000496
49,50,1709.66,0.88645,0.56344,1.00428,0.77241,0.65602,0.72702,0.47975,1.18220,0.81681,1.13996,0.000298,0.000298,0.000298


In [32]:
epochs_ran = df['epoch'].values if 'epoch' in df.columns else range(1, len(df)+1)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Training & Validation Curves', fontsize=14)

def safe_plot(ax, col, label, color, epochs):
    if col in df.columns:
        ax.plot(epochs, df[col], color=color, linewidth=1.5, label=label)
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, f'{col}\nnot found', ha='center', va='center')

# Box loss
ax = axes[0, 0]
safe_plot(ax, 'train/box_loss', 'train box loss', 'steelblue', epochs_ran)
safe_plot(ax, 'val/box_loss',   'val box loss',   'coral',     epochs_ran)
ax.set_title('Box Loss')

# Cls loss
ax = axes[0, 1]
safe_plot(ax, 'train/cls_loss', 'train cls loss', 'steelblue', epochs_ran)
safe_plot(ax, 'val/cls_loss',   'val cls loss',   'coral',     epochs_ran)
ax.set_title('Classification Loss')

# DFL loss
ax = axes[0, 2]
safe_plot(ax, 'train/dfl_loss', 'train dfl loss', 'steelblue', epochs_ran)
safe_plot(ax, 'val/dfl_loss',   'val dfl loss',   'coral',     epochs_ran)
ax.set_title('DFL Loss')

# mAP@0.5
ax = axes[1, 0]
safe_plot(ax, 'metrics/mAP50(B)', 'mAP@0.5', 'green', epochs_ran)
ax.axhline(0.60, color='red', linestyle='--', linewidth=1, label='Target 0.60')
ax.set_title('mAP@0.5')
ax.legend(fontsize=8)

# mAP@0.5:0.95
ax = axes[1, 1]
safe_plot(ax, 'metrics/mAP50-95(B)', 'mAP@0.5:0.95', 'purple', epochs_ran)
ax.set_title('mAP@0.5:0.95')

# Precision & Recall
ax = axes[1, 2]
safe_plot(ax, 'metrics/precision(B)', 'Precision', 'teal',   epochs_ran)
safe_plot(ax, 'metrics/recall(B)',    'Recall',    'orange', epochs_ran)
ax.set_title('Precision & Recall')

plt.tight_layout()
plt.savefig(run_dir / 'training_curves_custom.png', dpi=100)
plt.show()
print(f'Plot disimpan ke {run_dir}/training_curves_custom.png')

<Figure size 1600x800 with 6 Axes>

Plot disimpan ke runs\detect\runs\train\person_yolo11s_v1/training_curves_custom.png


## 3.8 Ringkasan Hasil Training

In [22]:
# Ambil nilai terbaik
best_map50_col = 'metrics/mAP50(B)'
best_map_col   = 'metrics/mAP50-95(B)'

best_map50    = df[best_map50_col].max() if best_map50_col in df.columns else None
best_map5095  = df[best_map_col].max()   if best_map_col   in df.columns else None
best_epoch    = df[best_map50_col].idxmax() + 1 if best_map50_col in df.columns else None

print('=' * 55)
print('          RINGKASAN HASIL TRAINING')
print('=' * 55)
print(f'Total epoch dijalankan : {len(df)}')
print(f'Epoch terbaik (val mAP): Epoch {best_epoch}')
print(f'Best val mAP@0.5       : {best_map50:.4f}  (target ≥ 0.60)')
print(f'Best val mAP@0.5:0.95  : {best_map5095:.4f}')
print(f'Status target          : {"TERCAPAI ✓" if best_map50 and best_map50 >= 0.60 else "BELUM TERCAPAI → lanjut tuning"}')
print(f'\nFile model terbaik     : {run_dir}/weights/best.pt')
print(f'File model terakhir    : {run_dir}/weights/last.pt')
print('=' * 55)

# Perbandingan dengan baseline
with open('runs/baseline_results.json') as f:
    baseline = json.load(f)

print(f'\nPerbandingan Baseline vs Fine-tuned:')
print(f'  mAP@0.5: {baseline["map50"]:.4f} → {best_map50:.4f} (Δ {best_map50 - baseline["map50"]:+.4f})')

# Simpan training summary
summary = {
    'run_name'      : RUN_NAME,
    'best_epoch'    : int(best_epoch) if best_epoch else None,
    'total_epochs'  : len(df),
    'best_map50'    : float(best_map50) if best_map50 else None,
    'best_map5095'  : float(best_map5095) if best_map5095 else None,
    'target_achieved': bool(best_map50 and best_map50 >= 0.60),
    'model_path'    : str(run_dir / 'weights' / 'best.pt')
}
with open('runs/training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('\nSummary tersimpan ke runs/training_summary.json')

          RINGKASAN HASIL TRAINING
Total epoch dijalankan : 50
Epoch terbaik (val mAP): Epoch 46
Best val mAP@0.5       : 0.7306  (target ≥ 0.60)
Best val mAP@0.5:0.95  : 0.4798
Status target          : TERCAPAI ✓

File model terbaik     : runs\detect\runs\train\person_yolo11s_v1/weights/best.pt
File model terakhir    : runs\detect\runs\train\person_yolo11s_v1/weights/last.pt

Perbandingan Baseline vs Fine-tuned:
  mAP@0.5: 0.8213 → 0.7306 (Δ -0.0908)

Summary tersimpan ke runs/training_summary.json


---
## ✅ Ringkasan Notebook 03

| Item | Detail |
|------|--------|
| Model | YOLOv11s (pretrained) |
| Epochs | Maks 50 (dengan early stopping) |
| Output | `runs/train/person_yolo11s_v1/weights/best.pt` |
| Monitoring | Loss curves, mAP curves tersimpan |

**Langkah selanjutnya** → `04_evaluation.ipynb`